# 04 · Gold features

Builds the model-ready dataset from silver.

**Input:** `kitsune_dev.silver.syn_dos_packets`

**Output:** `kitsune_dev.gold.syn_dos_features`, one row per packet with a `split` column, plus three views:
- `syn_dos_train`: normal traffic used to train the model
- `syn_dos_validation`: normal traffic used to set the alarm threshold
- `syn_dos_test`: the attack period, used once to measure detection

**Decisions (from the EDA):**
- Chronological split, never random
- Drop the 10 `hh_jit` mean and variance columns (leaked timestamps)

## Setup
**What:** reloads the `src` modules automatically and imports the gold functions.
**Why:** without autoreload, changes to `gold.py` are ignored until the notebook restarts.

In [0]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [0]:
from pyspark.sql import functions as F

from src.kitsune.gold import add_split, leaked_jitter_columns, validate_gold

## Config
**What:** table names and the split boundaries (row numbers).
**Why:** the values live here and are passed to the functions in `src/kitsune/gold.py`, so the logic works with any boundaries. Rows 1 to 2,682,349 have no attacks (EDA), so train and validation are split 80/20 inside that prefix. Test starts at the first attack.

In [0]:
# Tables
CATALOG = "kitsune_dev"
SILVER_PACKETS = f"{CATALOG}.silver.syn_dos_packets"
GOLD_FEATURES = f"{CATALOG}.gold.syn_dos_features"

# Split boundaries (row numbers), decided from the EDA.
# Train and validation are the attack-free prefix (rows 1 to 2,682,349), split 80/20.
# Test starts at the first attack
TRAIN_END_ROW = 2_145_879
VALIDATION_END_ROW = 2_682_349

## Load silver
**What:** reads the silver table and builds the list of the 10 leaked `hh_jit` columns.

In [0]:
silver = spark.table(SILVER_PACKETS)
leaked = leaked_jitter_columns()

## Leaked columns check
**What:** checks that the 10 leaked column names exist in silver.
**Why:** `df.drop()` ignores names that don't exist, so a typo would silently keep the leaked columns in gold.
**Result:** "All 10 leaked columns exist in silver" or an error listing the missing names.

In [0]:
# df.drop() ignores names that don't exist, so check them first
missing = []
for c in leaked:
    if c not in silver.columns:
        missing.append(c)

if missing:
    raise ValueError(f"Leaked columns not found in silver: {missing}")
print(f"All {len(leaked)} leaked columns exist in silver")

All 10 leaked columns exist in silver


## Build gold
**What:** adds the `split` column and drops the leaked columns.
**Result:** 117 columns in silver → 108 in gold (+1 `split`, −10 leaked).

In [0]:
# Tag every packet with its split, then remove the leaked columns
gold = add_split(silver, TRAIN_END_ROW, VALIDATION_END_ROW).drop(*leaked)
print(len(silver.columns), len(gold.columns))

117 108


## Split summary
**What:** rows and attacks per split, to see the result before validating it.

In [0]:
# Rows and attacks per split
split_summary = (
    gold.groupBy("split")
    .agg(
        F.count("*").alias("rows"),
        F.sum("label").alias("attacks"),
    )
    .orderBy("split")
)
display(split_summary)

split,rows,attacks
test,88927,7038
train,2145879,0
validation,536470,0


## Gold validation
**What:** runs `validate_gold` before writing anything.
**Rules:**
- No leaked column left in gold
- Every split has exactly the rows its boundaries say (each split, not just the total, so a moved boundary can't cancel out)
- No attacks in train or validation

**Result:** "Gold validation passed" or an error naming the broken rule.

In [0]:
validate_gold(gold, TRAIN_END_ROW, VALIDATION_END_ROW, silver.count())
print("Gold validation passed")

Gold validation passed
